# San Diego Regional Case Study

**Purpose**: Recreate the full analytical pipeline from the Wildfire Property Intelligence Report as a focused case study on **San Diego County** and its three neighboring counties: **Orange**, **Riverside**, and **Imperial**.

**Inputs:** checked-in result tables and optional source data. **Output:** `case_study_sd_region.json`. **Next:** the frontend consumes this JSON contract.

**Counties**: San Diego (6073), Orange (6059), Riverside (6065), Imperial (6025)


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial.distance import jensenshannon
import json
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('../../')
DATA_PATH = PROJECT_ROOT / 'dataset/Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz'
NEIGHBORS_PATH = PROJECT_ROOT / 'dataset/ca_county_neighbors.csv'
RESULTS_PATH = PROJECT_ROOT / 'results/tables'
OUTPUT_PATH = PROJECT_ROOT / 'website/frontend/public/data'

# San Diego region: SD + neighbors
SD_FIPS = '06073'
CASE_STUDY_FIPS = ['06025', '06059', '06065', '06073']  # Imperial, Orange, Riverside, San Diego
CASE_STUDY_FIPS_INT = [6025, 6059, 6065, 6073]

# FIPS helpers keep source and result-table identifiers aligned.
def fips_str(x):
    return str(x).zfill(5) if len(str(x)) < 5 else str(x)

def fips_int(x):
    return int(str(x).lstrip('0') or 0)

## 1. Load Data

In [2]:
# Optional source data; result tables provide the fallback.
try:
    df = pd.read_csv(DATA_PATH, low_memory=False)
    df['fips_str'] = df['fips'].apply(fips_str)
    df_region = df[df['fips'].isin(CASE_STUDY_FIPS_INT)].copy()
    print(f'Region rows: {len(df_region):,} of {len(df):,}')
except FileNotFoundError:
    df_region = None
    print('Dataset not found; using pre-computed results tables only.')

Region rows: 559,607 of 2,417,766


## 2. Load Pre-computed Results (filtered to case study counties)

In [3]:
# Bayesian shrinkage aggregated counts
bayesian = pd.read_csv(RESULTS_PATH / 'bayesian_shrinkage/bayesian_shrinkage_aggregated_counts.csv')
bayesian['fips_str'] = bayesian['fips'].astype(str).str.zfill(5)
bayesian_region = bayesian[bayesian['fips_str'].isin(CASE_STUDY_FIPS)].copy()

# Conditional probability summary and detail
cp_summary = pd.read_csv(RESULTS_PATH / 'conditional_probability/m01_neighbor_pool_county_lc_summary.csv')
cp_summary['fips_str'] = cp_summary['fips'].astype(str).str.zfill(5)
cp_summary_region = cp_summary[cp_summary['fips_str'].isin(CASE_STUDY_FIPS)].copy()

cp_detail = pd.read_csv(RESULTS_PATH / 'conditional_probability/m01_neighbor_pool_county_lc_color_detail.csv')
cp_detail['fips_str'] = cp_detail['fips'].astype(str).str.zfill(5)
cp_detail_region = cp_detail[cp_detail['fips_str'].isin(CASE_STUDY_FIPS)].copy()

# Group-level divergence
jsd_div = pd.read_csv(RESULTS_PATH / 'grouplevel_divergence/jsd_conditional_divergence.csv')
jsd_div['fips_str'] = jsd_div['fips'].astype(str).str.zfill(5)
jsd_div_region = jsd_div[jsd_div['fips_str'].isin(CASE_STUDY_FIPS)].copy()

jsd_summary = pd.read_csv(RESULTS_PATH / 'grouplevel_divergence/jsd_conditional_county_summary.csv')
jsd_summary['fips_str'] = jsd_summary['fips'].astype(str).str.zfill(5)
jsd_summary_region = jsd_summary[jsd_summary['fips_str'].isin(CASE_STUDY_FIPS)].copy()

# Exposure by county
exposure = pd.read_csv(RESULTS_PATH / '02_exposure_density_sparsity/eda_exposure_by_county.csv')
exposure['county_fips'] = exposure['county_fips'].astype(str).str.zfill(5)
exposure_region = exposure[exposure['county_fips'].isin(CASE_STUDY_FIPS)].copy()

print('Loaded all regional tables.')

Loaded all regional tables.


## 3. County Pairs: SD vs Each Neighbor (JSD)

In [4]:
# Load county-pair-comparisons for JSD between SD and neighbors
with open(OUTPUT_PATH / 'county-pair-comparisons.json') as f:
    pair_comparisons = json.load(f)

# SD neighbors: Imperial, Orange, Riverside
SD_NEIGHBORS = ['06025', '06059', '06065']
sd_pairs = {}
for nb in SD_NEIGHBORS:
    key1 = f'{SD_FIPS}-{nb}'
    key2 = f'{nb}-{SD_FIPS}'
    entry = pair_comparisons.get(key1) or pair_comparisons.get(key2)
    if entry:
        sd_pairs[key1] = entry

print('SD vs neighbor pairs:', list(sd_pairs.keys()))
for k, v in sd_pairs.items():
    print(f'  {k}: JSD = {v.get("jsd", {}).get("original", "?")}')

SD vs neighbor pairs: ['06073-06025', '06073-06059', '06073-06065']
  06073-06025: JSD = 0.5742742106311892
  06073-06059: JSD = 0.6836548456234984
  06073-06065: JSD = 0.6938442743749804


In [5]:
# Load post-pooling JSD (from color_pool greedy algorithm) if available
jsd_pooled = {}
pooled_path = OUTPUT_PATH / 'neighbor-jsd-pooled-greedy.json'
if pooled_path.exists():
    with open(pooled_path) as f:
        jsd_pooled = json.load(f)
    print('Loaded post-pooling JSD for', len(jsd_pooled), 'pairs')
    for k in sd_pairs:
        v = jsd_pooled.get(k) or jsd_pooled.get(k.split('-')[1] + '-' + k.split('-')[0])
        if v:
            print(f'  {k}: JSD pooled = {v["weighted_jsd"]:.4f}')
else:
    print('neighbor-jsd-pooled-greedy.json not found — run color_pool notebook first to export it')

Loaded post-pooling JSD for 144 pairs
  06073-06025: JSD pooled = 0.2262
  06073-06059: JSD pooled = 0.1994
  06073-06065: JSD pooled = 0.1960


## 4. Compute Surprisal from Conditional Probability Detail

In [6]:
# Surprisal: S = -log(p_pool). From m01 detail: p_pool is the pooled (neighbor) proportion.
cp_detail_region['surprisal'] = -np.log(np.clip(cp_detail_region['p_pool'], 1e-10, 1))

# Mean surprisal per county x landcover
surprisal_by_county_lc = cp_detail_region.groupby(['fips_str', 'lc_type']).agg(
    mean_surprisal=('surprisal', 'mean'),
    max_surprisal=('surprisal', 'max'),
    n_colors=('clr', 'nunique')
).reset_index()

# Mean surprisal per county (across landcovers)
surprisal_by_county = surprisal_by_county_lc.groupby('fips_str')['mean_surprisal'].mean().to_dict()
print('Mean surprisal by county:')
for f, s in surprisal_by_county.items():
    print(f'  {f}: {s:.3f}')

Mean surprisal by county:
  06025: 5.259
  06059: 3.282
  06065: 3.621
  06073: 4.770


## 5. Color Distributions P(color | landcover) per County

In [7]:
# From bayesian aggregated: aggregate to county x landcover x color proportions
county_lc_totals = bayesian_region.groupby(['fips_str', 'lc_type'])['count'].sum().reset_index()
county_lc_totals.rename(columns={'count': 'total'}, inplace=True)
dist = bayesian_region.merge(county_lc_totals, on=['fips_str', 'lc_type'])
dist['proportion'] = dist['count'] / dist['total']

# Build P(color|lc) per county for JSON
COUNTY_NAMES = {'06025': 'Imperial', '06059': 'Orange', '06065': 'Riverside', '06073': 'San Diego'}

distributions_by_county = {}
for fips in CASE_STUDY_FIPS:
    d = dist[dist['fips_str'] == fips]
    by_lc = {}
    for lc in d['lc_type'].unique():
        sub = d[d['lc_type'] == lc].sort_values('proportion', ascending=False)
        by_lc[lc] = [
            {'clr': row['clr'], 'proportion': round(row['proportion'], 4), 'count': int(row['count'])}
            for _, row in sub.iterrows()
        ]
    distributions_by_county[fips] = {
        'name': COUNTY_NAMES.get(fips, fips),
        'by_landcover': by_lc
    }

print('Color distributions built for', len(distributions_by_county), 'counties.')

Color distributions built for 4 counties.


## 6. Build Case Study JSON

In [8]:
case_study = {
    'counties': [
        {'fips': f, 'name': COUNTY_NAMES.get(f, f)}
        for f in CASE_STUDY_FIPS
    ],
    'exposure': {r['county_fips']: {'total_exposure': int(r['total_exposure']), 'median_exposure': float(r['median_exposure'])} for _, r in exposure_region.iterrows()},
    'distributions': distributions_by_county,
    'conditional_probability': {
        'summary': cp_summary_region.drop(columns=['fips_str'], errors='ignore').to_dict('records'),
        'mean_kl_by_county': cp_summary_region.groupby('fips_str')['kl_div'].mean().to_dict(),
        'mean_l1_by_county': cp_summary_region.groupby('fips_str')['l1_distance'].mean().to_dict(),
    },
    'surprisal': surprisal_by_county,
    'group_level_divergence': {
        'by_county_lc': jsd_div_region.drop(columns=['fips_str'], errors='ignore').to_dict('records'),
        'county_summary': jsd_summary_region.drop(columns=['fips_str'], errors='ignore').to_dict('records'),
        'avg_divergence_by_county': jsd_summary_region.set_index('fips_str')['avg_divergence'].to_dict(),
    },
    'sd_vs_neighbors': {
        k: {
            'county_a': v['county_a'],
            'county_b': v['county_b'],
            'jsd': {
                **(v.get('jsd', {})),
                **({'pooled': p} if (p := jsd_pooled.get(k) or jsd_pooled.get(k.split('-')[1] + '-' + k.split('-')[0])) else {})
            }
        }
        for k, v in sd_pairs.items()
    },
    'pair_keys': list(sd_pairs.keys()),
}

# Convert numpy types for JSON serialization
def convert(obj):
    if isinstance(obj, dict):
        return {k: convert(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert(x) for x in obj]
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    return obj

case_study = convert(case_study)

out_file = OUTPUT_PATH / 'case_study_sd_region.json'
# The frontend consumes this schema; keep keys and types stable.
with open(out_file, 'w') as f:
    json.dump(case_study, f, indent=2)

print(f'Saved to {out_file}')

Saved to ..\..\website\frontend\public\data\case_study_sd_region.json


## 7. Summary Stats for Report

In [9]:
print('=== San Diego Regional Case Study Summary ===')
print()
print('Exposure (total structures):')
for _, r in exposure_region.iterrows():
    print(f"  {COUNTY_NAMES.get(r['county_fips'], r['county_fips'])}: {r['total_exposure']:,}")
print()
print('JSD between San Diego and each neighbor:')
for k, v in sd_pairs.items():
    jsd_val = v.get('jsd', {}).get('original', 'N/A')
    print(f'  SD-{k.split("-")[1]}: {jsd_val}')
print()
print('Group-level avg divergence (vs statewide baseline):')
for _, r in jsd_summary_region.iterrows():
    print(f"  {COUNTY_NAMES.get(r['fips_str'], r['fips_str'])}: {r['avg_divergence']:.4f}")
print()
print('Mean surprisal (conditional probability):')
for f, s in surprisal_by_county.items():
    print(f"  {COUNTY_NAMES.get(f, f)}: {s:.3f}")

=== San Diego Regional Case Study Summary ===

Exposure (total structures):
  Orange: 784,918
  Riverside: 684,248
  San Diego: 843,891
  Imperial: 36,183

JSD between San Diego and each neighbor:
  SD-06025: 0.5742742106311892
  SD-06059: 0.6836548456234984
  SD-06065: 0.6938442743749804

Group-level avg divergence (vs statewide baseline):
  Imperial: 0.6083
  Orange: 0.4079
  Riverside: 0.4072
  San Diego: 0.5492

Mean surprisal (conditional probability):
  Imperial: 5.259
  Orange: 3.282
  Riverside: 3.621
  San Diego: 4.770
